# Single-image SAM + CLIP baseline

Run the cells from top to bottom in your existing **rttpc Jupyter kernel**. This notebook reuses the repository helpers without calling the full pipeline. It loads SAM and CLIP only; depth, mapping, and API scoring are separate stages. Initial model loading requires cached weights or Hugging Face access.

### 1. Imports and repository location
Find the checkout from either its root or the `notebooks` folder. Relative image paths below are resolved against that root. The source folder is added to Python's import search path; no installation is performed.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
from html import escape

cwd = Path.cwd().resolve()
repo_root = next(
    (p for p in (cwd, *cwd.parents) if (p / "src" / "rttpc" / "pipeline.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Open this notebook from the RTTPC checkout (root or notebooks folder).")
sys.path.insert(0, str(repo_root / "src"))

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from IPython.display import HTML, display
from rttpc.load_models import load_sam_model, load_clip_model
from rttpc.pipeline import generate_sam_masks, prepare_text_embeddings, score_with_clip

print(f"Repository: {repo_root}")
print(f"Torch: {torch.__version__}; CUDA available: {torch.cuda.is_available()}")
# Importing pipeline.py also imports Open3D, but no depth/map processing is run.


### 2. Choose and inspect the image
Edit `IMAGE_PATH` here. The supplied server candidate is the default; you can also use `Path("rock_in_path.png")` for a path relative to the repository root. A missing file stops execution before loading models.


In [ ]:
IMAGE_PATH = Path("/home/jovyan/RTTPC/rock_in_path.png")
image_path = IMAGE_PATH.expanduser()
if not image_path.is_absolute():
    image_path = repo_root / image_path
if not image_path.is_file():
    raise FileNotFoundError(f"Image not found: {image_path}. Edit IMAGE_PATH above.")

with Image.open(image_path) as source_image:
    image = source_image.convert("RGB")
print(f"Image: {image_path}; size: {image.width} x {image.height}")
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(image)
ax.set_title("Input image")
ax.axis("off")
plt.show()


### 3. Generate SAM masks
Load the existing SAM ViT-base checkpoint and run the repository's segmentation helper. These settings match its defaults: a 6×6 prompt grid, quality threshold 0.6, maximum area 40%, and batches of 16 points. The helper also filters tiny and duplicate masks. IDs are assigned in accepted-mask order and are local to this run.


In [ ]:
sam_model, sam_processor, sam_device = load_sam_model()
print(f"SAM device: {sam_device}")
segmentations = generate_sam_masks(
    image, sam_model, sam_processor, sam_device,
    grid_size=6,
    mask_quality_threshold=0.6,
    max_area_ratio=0.4,
    points_per_batch=16,
)
for mask_id, segment in enumerate(segmentations, start=1):
    segment["mask_id"] = mask_id
print(f"Accepted SAM masks: {len(segmentations)}")
if not segmentations:
    print("No masks accepted. CLIP scoring will be skipped; inspect the image and SAM settings.")


### 4. Score masks with CLIP
The prompts below match `pipeline()` exactly. CLIP scores the padded rectangular crop around each mask, including background inside that rectangle.

Let **p** be the maximum positive cosine similarity and **n** the maximum negative similarity. The unchanged formula is `100 * (p if p > n else -n)`. Positive scores generally indicate traversable content; negative scores generally indicate hazards. These are similarity-based scores, not probabilities. The baseline's selection threshold is **score > 0.01**, after scaling by 100.

Invalid results (`None`, nonnumeric values, NaN, or infinity) are marked explicitly and excluded from selection. Check the helper's logged error when a score is invalid.


In [ ]:
positive_queries = [
    "a photo of a flat clear road",
    "a photo of a walkable dirt path",
    "a photo of flat grassy ground",
]
negative_queries = [
    "a photo of the sky",
    "a photo of a wall",
    "a photo of deep water",
    "a photo of a puddle",
    "a photo of a tree",
    "a photo of trees",
    "a photo of a large rock",
    "a photo of a person",
    "a photo of a cliff",
    "a photo of a bush",
]
TRAVERSABLE_THRESHOLD = 0.01
results = []
if segmentations:
    clip_model, clip_processor, clip_device = load_clip_model()
    print(f"CLIP device: {clip_device}")
    positive_embeddings, negative_embeddings = prepare_text_embeddings(
        positive_queries, negative_queries, clip_model, clip_processor, clip_device,
    )
    for segment in segmentations:
        raw_score = score_with_clip(
            image, segment["mask"], clip_model, clip_processor, clip_device,
            positive_embeddings, negative_embeddings,
        )
        try:
            raw_score = float(raw_score)
            score = 100.0 * raw_score if np.isfinite(raw_score) else None
            if score is not None and not np.isfinite(score):
                score = None
        except (TypeError, ValueError, OverflowError):
            score = None
        segment["trav_score"] = score
        status = (
            "invalid" if score is None else
            "selected" if score > TRAVERSABLE_THRESHOLD else "not selected"
        )
        results.append({
            "mask_id": segment["mask_id"],
            "area": segment["area"],
            "sam_confidence": segment["confidence"],
            "score": score,
            "status": status,
        })
        if score is None:
            print(f"Mask {segment['mask_id']}: invalid CLIP score (see logged errors).")
    print(f"Scored masks: {len(results)}; invalid: {sum(r['score'] is None for r in results)}")
else:
    print("CLIP skipped: there are no accepted masks.")


### 5. Results table
Each row links a mask ID to its area, SAM quality estimate, scaled CLIP score, and baseline selection decision. “Not selected” means below the baseline threshold; it is not a calibrated assessment of robot safety.


In [ ]:
if results:
    rows = []
    for result in results:
        score_text = "invalid" if result["score"] is None else f"{result['score']:.3f}"
        rows.append(
            f"<tr><td>{result['mask_id']}</td><td>{result['area']}</td>"
            f"<td>{result['sam_confidence']:.3f}</td><td>{score_text}</td>"
            f"<td>{escape(result['status'])}</td></tr>"
        )
    display(HTML(
        "<table><thead><tr><th>Mask ID</th><th>Area (pixels)</th>"
        "<th>SAM confidence</th><th>CLIP score ×100</th><th>Status</th>"
        "</tr></thead><tbody>" + "".join(rows) + "</tbody></table>"
    ))
else:
    print("No result rows: SAM accepted zero masks.")


### 6. Inline mask overlays
The overview labels every accepted mask with its ID and score (green = selected, red = not selected, gray = invalid). Overlapping masks can obscure each other, so the following panels show one mask at a time. No native Open3D window is opened.


In [ ]:
def draw_segment(ax, segment):
    score = segment["trav_score"]
    color = (
        (0.5, 0.5, 0.5) if score is None else
        (0.1, 0.8, 0.2) if score > TRAVERSABLE_THRESHOLD else (0.9, 0.15, 0.1)
    )
    mask = segment["mask"]
    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    overlay[mask] = (*color, 0.4)
    ax.imshow(overlay)
    x, y = segment["point"]
    score_text = "invalid" if score is None else f"{score:+.2f}"
    ax.text(x, y, f"#{segment['mask_id']}: {score_text}", color="white",
            fontsize=9, bbox=dict(facecolor="black", alpha=0.7, pad=2))
    return score_text

if segmentations:
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(image)
    for segment in segmentations:
        draw_segment(ax, segment)
    ax.set_title("All accepted masks: ID and scaled CLIP score")
    ax.axis("off")
    plt.show()

    # Limit each figure to six masks to keep individual labels readable.
    for start in range(0, len(segmentations), 6):
        batch = segmentations[start:start + 6]
        fig, axes = plt.subplots(
            (len(batch) + 2) // 3, 3,
            figsize=(15, 4 * ((len(batch) + 2) // 3)),
            squeeze=False,
        )
        for ax in axes.flat:
            ax.axis("off")
        for ax, segment in zip(axes.flat, batch):
            ax.imshow(image)
            score_text = draw_segment(ax, segment)
            ax.set_title(f"Mask #{segment['mask_id']} — score {score_text}")
        fig.tight_layout()
        plt.show()
else:
    print("No mask overlays to display.")
